# Credit Card Fraud Detection
Using imbalance handling, SMOTE and classification models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [ ]:
df = pd.read_csv('data/creditcard.csv')
print(df.shape)
df.head()

In [ ]:
print(df['Class'].value_counts())
print((df['Class'].value_counts(normalize=True)*100).round(4))

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']
scaler = StandardScaler()
X[['Time','Amount']] = scaler.fit_transform(X[['Time','Amount']])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(y_train.value_counts())

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.08, eval_metric='logloss', random_state=42, n_jobs=-1)
}

results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]
    results.append([name, precision_score(y_test,pred,zero_division=0), recall_score(y_test,pred,zero_division=0), f1_score(y_test,pred,zero_division=0), roc_auc_score(y_test,prob), average_precision_score(y_test,prob)])
    print('\n', name)
    print(classification_report(y_test,pred,zero_division=0))

pd.DataFrame(results, columns=['Model','Precision','Recall','F1','ROC-AUC','PR-AUC'])